# DeepSORT Final Project — Google Colab

**Runtime:** GPU (T4)  
**Данные:** `MyDrive/Videos-CV/` (6 MOT-последовательностей)  
**Репозиторий:** [danvlak-batya/deep-sort-project](https://github.com/danvlak-batya/deep-sort-project)

## Порядок выполнения
1. Клон репозитория и установка зависимостей
2. Подключение Google Drive
3. Трекинг на одном видео (проверка)
4. Трекинг на всех 6 видео
5. Overlay-видео
6. Оценка HOTA (TrackEval)

> Не используйте `pip install -r requirements.txt` — в Colab ставьте зависимости через `tools/install_colab_deps.py`.

In [ ]:
!nvidia-smi

## 1. Клон репозитория и установка

In [ ]:
REPO_URL = "https://github.com/danvlak-batya/deep-sort-project.git"
PROJECT_DIR = "/content/deep-sort-project"

import os
os.chdir("/content")
if os.path.exists(PROJECT_DIR):
    !rm -rf {PROJECT_DIR}

!git clone {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}

!python tools/install_colab_deps.py

## 2. Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA_ROOT = "/content/drive/MyDrive/Videos-CV"
import os
assert os.path.isdir(DATA_ROOT), f"Папка не найдена: {DATA_ROOT}"
print("Последовательности:", os.listdir(DATA_ROOT))

## 3. Настройки моделей

In [ ]:
DETECTOR = "yolov8n"      # yolov8n | yolov5s | rtdetr_r18
REID = "osnet_x0_25"      # osnet_x0_25 | resnet50_ibn | fastreid_sbs
CONFIG = "configs/default.yaml"
RESULTS_DIR = "results/demo"

## 4. Трекинг — одно видео (проверка)

Папка на Drive может называться иначе (например `Kitti-17`), имя результата — стандартное (`KITTI-17.txt`).

In [ ]:
from utils.mot_paths import find_sequence_dir, list_image_filenames

SEQUENCE_FOLDER = "Kitti-17"   # имя папки на Drive
SEQUENCE = "KITTI-17"          # имя для файла результатов

SEQ_DIR = find_sequence_dir(DATA_ROOT, SEQUENCE_FOLDER)
print(f"Кадров: {len(list_image_filenames(SEQ_DIR))}")

import os
os.makedirs(RESULTS_DIR, exist_ok=True)

!python run_tracker.py \
  --sequence_dir "{SEQ_DIR}" \
  --config {CONFIG} \
  --detector {DETECTOR} \
  --reid {REID} \
  --output_file {RESULTS_DIR}/{SEQUENCE}.txt

## 5. Трекинг — все 6 видео

In [ ]:
!python eval/run_benchmark.py \
  --mot_root "{DATA_ROOT}" \
  --output_dir {RESULTS_DIR} \
  --config {CONFIG} \
  --detector {DETECTOR} \
  --reid {REID}

## 6. Overlay-видео

In [ ]:
!python tools/generate_overlays.py \
  --mot_dir "{DATA_ROOT}" \
  --results_dir {RESULTS_DIR} \
  --output_dir results/overlays \
  --sequence {SEQUENCE}

from IPython.display import Video, display
overlay_path = f"results/overlays/{SEQUENCE}_overlay.mp4"
if os.path.exists(overlay_path):
    display(Video(overlay_path, embed=True))
else:
    print("Видео не найдено:", overlay_path)

## 7. Оценка HOTA

Требуются `.txt` файлы треков в `results/demo/` для всех 6 последовательностей.

Если появляется `ImportError: _center` — **Runtime → Restart session** и выполните ячейки с начала (numpy/scipy должны ставиться вместе).

In [ ]:
from eval.mot_metrics import evaluate_hota, EVAL_SEQUENCES

metrics = evaluate_hota(DATA_ROOT, RESULTS_DIR)

print("\n=== HOTA results ===")
print(f"{'Sequence':<16} {'HOTA':>8}")
print("-" * 26)
for seq in EVAL_SEQUENCES:
    if seq in metrics["per_sequence"]:
        hota = metrics["per_sequence"][seq]["HOTA"]
        print(f"{seq:<16} {hota:8.4f}")
    else:
        print(f"{seq:<16}   (нет данных)")
print("-" * 26)
print(f"{'Mean':<16} {metrics['mean_hota']:8.4f}")